
# Deepfake Detection Challenge Baseline


- **ImageNet pretrained backbone 1개**
- **GAN 기반 얼굴 데이터셋 1개** 사용
- 최종적으로 `submission_baseline.csv` 생성

---

## 목차

1. 환경 설정  
2. 경로 및 하이퍼파라미터 설정  
3. 대회 데이터 압축 해제  
4. 학습 데이터 다운로드 및 로드  
5. 학습/검증 데이터셋 구성  
6. 모델 정의  
7. 학습  
8. 검증  
9. 테스트 추론 및 제출 파일 생성  

---

## 설계

- backbone: `torchvision`의 `ResNet18`
- backbone 대부분 **freeze**
- 외부 데이터는 **GAN 기반 얼굴 데이터셋 1개만** 사용
- 학습 데이터는 전체를 다 쓰지 않고 **작은 subset만 사용**




## 학습 데이터셋

이 베이스라인은 **Kaggle의 `xhlulu/140k-real-and-fake-faces`** 데이터셋을 사용합니다.

- 총 **140,000장**
- **real 70,000장 / fake 70,000장**
- fake 이미지는 **StyleGAN 기반 GAN 얼굴**로 구성되어 있습니다.

이 노트북에서는 라벨을 다음과 같이 사용합니다.

- **real = 0**
- **fake = 1**

즉, `real/` 폴더 이미지는 모두 `0`, `fake/` 폴더 이미지는 모두 `1`로 사용합니다.


In [1]:
# ============================================================
# 1. 필수 패키지 설치
# ============================================================

!pip cache purge
!pip install --user kagglehub pillow==11.1.0 --ignore-installed


Files removed: 72 (7.9 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 11.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 19.4 MB/s eta 0:00:00
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/lightning_utilities-0.14.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/nvfuser-0.2.26a0+c5e1555-py3.12-linux-x86_64.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/bitsandbytes-0.48.0.dev0-py3.12-linux-x86_64.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement i

In [2]:
# ============================================================
# 2. 라이브러리 / 경로 / 하이퍼파라미터 설정
# ============================================================
import os
import zipfile
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision
from torchvision import transforms, datasets

import kagglehub

ImageFile.LOAD_TRUNCATED_IMAGES = True

# -------------------------------
# 재현성
# -------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# -------------------------------
# 경로
# -------------------------------
WORKING_DIR = Path(os.getcwd())

COMP_ZIP_PATH = "deepfake-detection-challenge-deep-preview-x-ai-7th.zip"
EXTRACT_DIR = Path("competition_data")

DATASET_SLUG = "xhlulu/140k-real-and-fake-faces"
DATASET_DIR = None  # download 후 자동 채움

# -------------------------------
# 학습 설정
# -------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2

TRAIN_PER_CLASS = 250
VAL_PER_CLASS = 80

EPOCHS = 10
LR = 1e-3

print("DEVICE:", DEVICE)


DEVICE: cuda


In [3]:
# ============================================================
# 3. 대회 데이터 압축 해제
# ============================================================
assert Path(COMP_ZIP_PATH).exists(), f"압축 파일이 없습니다: {COMP_ZIP_PATH}"

if EXTRACT_DIR.exists():
    import shutil
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(COMP_ZIP_PATH, "r") as zf:
    zf.extractall(EXTRACT_DIR)

print("압축 해제 완료:", EXTRACT_DIR)
print("파일 목록:", os.listdir(EXTRACT_DIR))


압축 해제 완료: competition_data
파일 목록: ['sample_submission.csv', 'test.csv', 'test']


In [4]:
# ============================================================
# 4. 테스트 데이터와 제출 형식 불러오기
# ============================================================
TEST_DIR = EXTRACT_DIR / "test"
TEST_CSV_PATH = EXTRACT_DIR / "test.csv"
SAMPLE_SUB_PATH = EXTRACT_DIR / "sample_submission.csv"

assert TEST_DIR.exists(), "test 폴더가 없습니다."
assert TEST_CSV_PATH.exists(), "test.csv가 없습니다."
assert SAMPLE_SUB_PATH.exists(), "sample_submission.csv가 없습니다."

test_df = pd.read_csv(TEST_CSV_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

print(test_df.head())
print(sample_sub.head())
print("테스트 이미지 수:", len(test_df))

        id         file_path
0  test001  test/test001.png
1  test002  test/test002.png
2  test003  test/test003.png
3  test004  test/test004.png
4  test005  test/test005.png
        id  score
0  test001    0.5
1  test002    0.5
2  test003    0.5
3  test004    0.5
4  test005    0.5
테스트 이미지 수: 300



## Kaggle 데이터셋 다운로드

이 셀은 **GAN 기반 얼굴 데이터셋**을 내려받습니다.  
Colab에서 처음 실행하면 Kaggle 인증이 필요할 수 있습니다.


In [5]:
# ============================================================
# 5. 학습 데이터 다운로드
# ============================================================
# xhlulu/140k-real-and-fake-faces
# - real/ 70k
# - fake/ 70k (StyleGAN 계열)
DATASET_DIR = Path(kagglehub.dataset_download(DATASET_SLUG))
print("Downloaded dataset to:", DATASET_DIR)

# 데이터셋 내부 구조 확인
for p in DATASET_DIR.iterdir():
    print(p.name)

Downloaded dataset to: /home/work/.cache/kagglehub/datasets/xhlulu/140k-real-and-fake-faces/versions/2
valid.csv
test.csv
real_vs_fake
train.csv


In [6]:
# ============================================================
# 6. 이미지 전처리 정의
# ============================================================
train_tfms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

valid_tfms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [7]:
# ============================================================
# 7. 학습 데이터셋 구성
#    ImageFolder 구조를 사용합니다.
#    기대 폴더:
#      real/
#      fake/
# ============================================================
# 데이터셋 루트 자동 탐색
candidate_roots = [DATASET_DIR]
for sub in DATASET_DIR.rglob("*"):
    if sub.is_dir():
        child_names = sorted([p.name.lower() for p in sub.iterdir() if p.is_dir()])
        if "real" in child_names and "fake" in child_names:
            candidate_roots.append(sub)

dataset_root = None
for c in candidate_roots:
    child_names = sorted([p.name.lower() for p in c.iterdir() if p.is_dir()])
    if "real" in child_names and "fake" in child_names:
        dataset_root = c
        break

assert dataset_root is not None, "real/ fake/ 폴더 구조를 찾지 못했습니다."
print("Using dataset root:", dataset_root)

full_train_ds = datasets.ImageFolder(root=str(dataset_root), transform=train_tfms)
full_valid_ds = datasets.ImageFolder(root=str(dataset_root), transform=valid_tfms)

print("class_to_idx:", full_train_ds.class_to_idx)

Using dataset root: /home/work/.cache/kagglehub/datasets/xhlulu/140k-real-and-fake-faces/versions/2/real_vs_fake/real-vs-fake/test
class_to_idx: {'fake': 0, 'real': 1}


In [8]:
# ============================================================
# 8. 라벨 매핑 정리
#    최종 학습 라벨은 real=0, fake=1 로 맞춥니다.
# ============================================================
class_to_idx = full_train_ds.class_to_idx

def original_to_target_label(orig_label: int):
    # ImageFolder의 class 인덱스를 우리 target label로 변환
    # fake -> 1, real -> 0
    class_name = [k for k, v in class_to_idx.items() if v == orig_label][0].lower()
    if class_name == "fake":
        return 1
    elif class_name == "real":
        return 0
    else:
        raise ValueError(f"Unknown class name: {class_name}")

# 클래스별 인덱스 모으기
real_indices = []
fake_indices = []

for i, (_, y) in enumerate(full_train_ds.samples):
    class_name = [k for k, v in class_to_idx.items() if v == y][0].lower()
    if class_name == "real":
        real_indices.append(i)
    elif class_name == "fake":
        fake_indices.append(i)

random.Random(SEED).shuffle(real_indices)
random.Random(SEED).shuffle(fake_indices)

train_real_idx = real_indices[:TRAIN_PER_CLASS]
val_real_idx = real_indices[TRAIN_PER_CLASS:TRAIN_PER_CLASS + VAL_PER_CLASS]

train_fake_idx = fake_indices[:TRAIN_PER_CLASS]
val_fake_idx = fake_indices[TRAIN_PER_CLASS:TRAIN_PER_CLASS + VAL_PER_CLASS]

print("train real:", len(train_real_idx))
print("train fake:", len(train_fake_idx))
print("val real:", len(val_real_idx))
print("val fake:", len(val_fake_idx))

train real: 250
train fake: 250
val real: 80
val fake: 80


In [9]:
# ============================================================
# 9. 라벨 변환용 Wrapper Dataset
# ============================================================
class RemapLabelSubset(Dataset):
    def __init__(self, base_dataset, indices):
        self.base_dataset = base_dataset
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        x, orig_y = self.base_dataset[self.indices[idx]]
        y = torch.tensor(float(original_to_target_label(int(orig_y))), dtype=torch.float32)
        return x, y

train_dataset = torch.utils.data.ConcatDataset([
    RemapLabelSubset(full_train_ds, train_real_idx),
    RemapLabelSubset(full_train_ds, train_fake_idx),
])

val_dataset = torch.utils.data.ConcatDataset([
    RemapLabelSubset(full_valid_ds, val_real_idx),
    RemapLabelSubset(full_valid_ds, val_fake_idx),
])

print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))

Train size: 500
Val size: 160


In [10]:
# ============================================================
# 10. DataLoader
# ============================================================
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

In [11]:
# ============================================================
# 11. 모델 정의
# - ResNet18 backbone
# - backbone freeze
# - 마지막 fc만 학습
# ============================================================
model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)

for p in model.parameters():
    p.requires_grad = False

in_features = model.fc.in_features
model.fc = nn.Linear(in_features, 1)

model = model.to(DEVICE)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=LR)

print(model.fc)

Linear(in_features=512, out_features=1, bias=True)


In [12]:
# ============================================================
# 12. ROC-AUC 계산 함수
# ============================================================
from sklearn.metrics import roc_auc_score

@torch.no_grad()
def evaluate_auc(model, loader):
    model.eval()
    all_probs = []
    all_labels = []

    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        logits = model(x).squeeze(1)
        probs = torch.sigmoid(logits)

        all_probs.extend(probs.detach().cpu().numpy().tolist())
        all_labels.extend(y.detach().cpu().numpy().tolist())

    auc = roc_auc_score(all_labels, all_probs)
    return auc

In [13]:
# ============================================================
# 13. 학습
# ============================================================
for epoch in range(EPOCHS): #<-epoch만큼 반복학습
    model.train()
    running_loss = 0.0

    for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad()

        logits = model(x).squeeze(1)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x.size(0)

    train_loss = running_loss / len(train_loader.dataset)
    val_auc = evaluate_auc(model, val_loader)

    print(f"[Epoch {epoch+1}] train_loss={train_loss:.4f} | val_auc={val_auc:.4f}")

import gc
gc.collect()
torch.cuda.empty_cache()

Epoch 1/10:   0%|          | 0/16 [00:00<?, ?it/s]

[Epoch 1] train_loss=0.6886 | val_auc=0.6698


Epoch 2/10:   0%|          | 0/16 [00:00<?, ?it/s]

[Epoch 2] train_loss=0.6446 | val_auc=0.8050


Epoch 3/10:   0%|          | 0/16 [00:00<?, ?it/s]

...

Epoch 9/10:   0%|          | 0/16 [00:00<?, ?it/s]

[Epoch 9] train_loss=0.5034 | val_auc=0.8728


Epoch 10/10:   0%|          | 0/16 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f72bfa26160>Exception ignored in: 
<function _MultiProcessingDataLoaderIter.__del__ at 0x7f72bfa26160>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1662, in __del__
    Traceback (most recent call last):
self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1662, in __del__
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1645, in _shutdown_workers
        if w.is_alive():
self._shutdown_workers()  
     File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1645, in _shutdown_workers
  ^^^    ^^^^if w.is_alive():^^^
^^ 
   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
      assert self._parent_pid == os.getpid(), 'can only test a child process' 
     ^^    ^ ^   ^^^^^^^^^^^^^^^^^^^^^

[Epoch 10] train_loss=0.5009 | val_auc=0.8720


In [14]:
# ============================================================
# 14. 대회 테스트셋 Dataset
# ============================================================
class CompetitionTestDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.root_dir = Path(root_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.root_dir / f"{row['id']}.png"
        img = Image.open(img_path).convert("RGB")

        if self.transform is not None:
            img = self.transform(img)

        return row["id"], img

test_dataset = CompetitionTestDataset(test_df, TEST_DIR, transform=valid_tfms)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

In [15]:
# ============================================================
# 15. 테스트셋 추론
# ============================================================
@torch.no_grad()
def predict_test(model, loader):
    model.eval()
    ids = []
    probs = []

    for batch_ids, x in tqdm(loader, desc="Predicting test set"):
        x = x.to(DEVICE)
        logits = model(x).squeeze(1)
        batch_probs = torch.sigmoid(logits).detach().cpu().numpy().tolist()

        ids.extend(list(batch_ids))
        probs.extend(batch_probs)

    return ids, probs

pred_ids, pred_scores = predict_test(model, test_loader)

Predicting test set:   0%|          | 0/10 [00:00<?, ?it/s]

In [16]:
# ============================================================
# 16. 제출 파일 생성
# ============================================================
submission = pd.DataFrame({
    "id": pred_ids,
    "score": pred_scores
})

submission = sample_sub[["id"]].merge(submission, on="id", how="left")

submission_path = "competition_data/sample_submission.csv"
submission.to_csv(submission_path, index=False)

print("Saved submission to:", submission_path)
submission.head()

Saved submission to: competition_data/sample_submission.csv


,id,score
0,test001,0.467721
1,test002,0.260263
2,test003,0.316663
3,test004,0.142005
4,test005,0.347403
